# Laboratorio 4: Optimización Numérica y Respuesta Transitoria
## Resolución de Circuitos de Segundo Orden (RLC) mediante EDOs

---

### Objetivo del Laboratorio

En esta experiencia vamos a programar y visualizar la **respuesta transitoria** de un circuito RLC conectado en serie a una fuente DC de $V_s = 12$ V que se enciende en $t=0$.

Para ello, aplicaremos la Ley de Kirchhoff y modelaremos la física del circuito en forma de un **sistema de ecuaciones diferenciales de primer orden**, que luego resolveremos utilizando el módulo `solve_ivp` de SciPy.

---

### 1. Modelo Matemático (De Teoría a Código)

La ecuación fundamental de la malla (sumatoria de caídas de tensión) nos dicta que:

$$V_L(t) + V_R(t) + V_C(t) = V_s$$

Expresando el voltaje del inductor y resistor en función de la corriente, y recordando que $i = C \frac{dV_C}{dt}$, llegamos a nuestra Ecuación Diferencial Ordinaria (EDO) de segundo orden:

$$LC \frac{d^2 V_C}{dt^2} + RC \frac{dV_C}{dt} + V_C = V_s$$

Dado que las herramientas numéricas resuelven sistemas de primer orden, aplicamos variables de estado:
- $y_0 = V_C$ (Tensión del capacitor)
- $y_1 = \frac{dV_C}{dt}$ (Tasa de cambio del voltaje)

Despejando obtenemos el sistema que introduciremos en Python:

$$\dot{y}_0 = y_1$$
$$\dot{y}_1 = \frac{V_s - y_0}{LC} - \frac{R}{L} y_1$$

### 2. Importación de Librerías
Asegúrate de ejecutar esta celda antes de continuar con la programación.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

### 3. Programación de la Función del Sistema

Creamos la función que representará nuestro circuito matemático. A diferencia de ejemplos anteriores, haremos que la resistencia `R` sea un parámetro de la función (agregándolo a través del argumento `args` de `solve_ivp`). Esto nos permitirá reutilizar la misma función para los tres casos de amortiguamiento.

In [ ]:
# Parámetros físicos constantes del circuito
L = 0.5       # Inductancia (Henrios)
C = 2e-6      # Capacitancia (Faradios) -> 2 uF
Vs = 12.0     # Voltaje de la fuente DC (Voltios)

def circuito_rlc(t, y, R):
    """
    Representación en variables de estado del circuito RLC serie.
    
    t: Tiempo (requerido por la firma de solve_ivp, aunque el sistema es autónomo)
    y: Arreglo de estado actual [V_c, dV_c/dt]
    R: Valor de la resistencia en Ohmios (pasado por args)
    
    Retorna:
    Lista con las derivadas [dy0/dt, dy1/dt]
    """
    y0, y1 = y
    
    dy0_dt = y1
    dy1_dt = (Vs - y0) / (L * C) - (R / L) * y1
    
    return [dy0_dt, dy1_dt]

# Condiciones Iniciales en t = 0
# El capacitor está completamente descargado (0V) y no hay corriente (0 V/s de cambio).
y0_inicial = [0.0, 0.0]

### 4. Simulación: Los Tres Regímenes del Transitorio

El comportamiento del voltaje dependerá enteramente de cuánta resistencia tenga el circuito. Simularemos tres valores de $R$ que provocarán comportamientos físicos drásticamente diferentes.

- **Subamortiguado (R pequeña):** La energía oscila entre el capacitor y el inductor antes de disiparse.
- **Crítico (R exacta = $2\sqrt{L/C}$):** El sistema llega al equilibrio lo más rápido posible sin rebotar ni oscilar.
- **Sobreamortiguado (R muy grande):** La resistencia 'frena' tanto el flujo de energía que el capacitor tarda muchísimo en cargarse.

In [ ]:
# Calculamos la resistencia crítica teórica
R_critica = 2 * np.sqrt(L / C)

# Nuestros 3 casos de estudio para R:
R_sub     = R_critica * 0.1   # Muy poca fricción eléctrica -> oscilará
R_crit    = R_critica         # El punto perfecto
R_sobre   = R_critica * 5.0   # Demasiada fricción eléctrica -> lentitud

print(f"Resistencia Subamortiguada:  {R_sub:.1f} Ohms")
print(f"Resistencia Crítica:         {R_crit:.1f} Ohms")
print(f"Resistencia Sobreamortiguada: {R_sobre:.1f} Ohms")

# Vector de tiempo: Simularemos los primeros 0.05 segundos con alta resolución
t_span = (0.0, 0.05)
t_eval = np.linspace(0, 0.05, 2000)

# Resolvemos los tres sistemas usando RK45 (Runge-Kutta)
sol_sub   = solve_ivp(circuito_rlc, t_span, y0_inicial, args=(R_sub,),   t_eval=t_eval, method='RK45')
sol_crit  = solve_ivp(circuito_rlc, t_span, y0_inicial, args=(R_crit,),  t_eval=t_eval, method='RK45')
sol_sobre = solve_ivp(circuito_rlc, t_span, y0_inicial, args=(R_sobre,), t_eval=t_eval, method='RK45')

### 5. Visualización del Comportamiento Transitorio

A continuación, extraemos el voltaje $V_C$ (que es nuestra variable $y_0$ en la salida de `solve_ivp`) y graficamos los resultados obtenidos en el tiempo.

In [ ]:
plt.figure(figsize=(12, 7))

# Extraemos la primera fila (.y[0]) que corresponde al voltaje en el capacitor
plt.plot(sol_sub.t * 1000,   sol_sub.y[0],   color='tomato',       lw=2, label='Subamortiguado ($R=100\ \Omega$)')
plt.plot(sol_crit.t * 1000,  sol_crit.y[0],  color='mediumseagreen', lw=2, label='Crítico ($R=1000\ \Omega$)')
plt.plot(sol_sobre.t * 1000, sol_sobre.y[0], color='steelblue',    lw=2, label='Sobreamortiguado ($R=5000\ \Omega$)')

# Línea de referencia del voltaje de la fuente (estado estacionario final)
plt.axhline(Vs, color='k', linestyle='--', alpha=0.6, label='Voltaje Fuente (12V)')

# Detalles estéticos de la gráfica
plt.title('Respuesta Transitoria del Voltaje $V_C(t)$ en el Circuito RLC', fontsize=14, fontweight='bold')
plt.xlabel('Tiempo [ms]', fontsize=12)
plt.ylabel('Voltaje del Capacitor $V_C$ [V]', fontsize=12)
plt.legend(fontsize=10, loc='lower right')
plt.grid(True, alpha=0.4)
plt.xlim(0, 50)

plt.tight_layout()
plt.show()

### Conclusiones Matemáticas y Físicas

1. **Sobreimpulso (Overshoot):** Observa cómo el caso subamortiguado supera los 12V, llegando a casi 20V en el primer pico. Esto ocurre porque el inductor sigue inyectando corriente al capacitor incluso cuando este ya alcanzó los 12V. Es un comportamiento vital a tener en cuenta al diseñar circuitos, ¡pues el exceso de voltaje podría quemar componentes!
2. **Velocidad de carga:** El amortiguamiento crítico alcanza y se asienta en el valor de los 12V más rápido que cualquier otro diseño sin sufrir rebotes, por lo que suele ser el objetivo central al optimizar la respuesta de un sistema físico.